# Session 25: Tool Calling 파인튜닝 with Qwen2.5-3B (LoRA)

## 🎯 Tool Calling 파인튜닝 실습

이번 세션에서는 이전에 생성한 Tool Calling 데이터를 사용하여 **Qwen2.5-3B 모델에 Tool Calling 능력을 부여**합니다.

### 실습 흐름

```
데이터 로드 → 포맷팅 → LoRA 설정 → 학습 → Before/After 비교 → 다양한 테스트
```

### 실습 환경

| 항목 | 설정 |
|------|------|
| 모델 | Qwen2.5-3B-Instruct (4bit) |
| 방법 | QLoRA (r=16) |
| 데이터 | Tool Calling 샘플 데이터 |
| GPU | RTX 4060 (8GB VRAM) |

### 학습 목표

- ✅ Tool Calling 데이터를 Chat Template으로 변환
- ✅ QLoRA로 3B 모델 파인튜닝
- ✅ 학습 전후 Tool Calling 능력 비교
- ✅ 다양한 시나리오에서 도구 호출 테스트

## 1️⃣ 환경 설정 및 Qwen2.5-3B 로드 (4bit)

In [ ]:
# 필수 라이브러리
import torch
import gc
import os
import json
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print_gpu_memory("시작")

In [ ]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "./output/tool_calling_finetuning"
MAX_SEQ_LENGTH = 1024

# 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 토크나이저 로드
print("⏳ 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ 토크나이저 로드 완료 (vocab: {tokenizer.vocab_size:,})")

In [ ]:
# 모델 로드
print("⏳ 모델 로딩 중... (4bit 양자화)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.gradient_checkpointing_enable()

print(f"✅ 모델 로드 완료")
print(f"📌 모델: {MODEL_NAME}")
print(f"📌 파라미터: {model.num_parameters():,}")
print_gpu_memory("모델 로드 후")

## 2️⃣ Tool Calling 데이터 로드 및 포맷팅

In [ ]:
# Tool Calling 데이터 로드
data_path = "../data/samples/tool_calling_sample.json"

with open(data_path, "r", encoding="utf-8") as f:
    tool_data = json.load(f)

print(f"✅ 데이터 로드 완료: {len(tool_data)}개")
print(f"📌 경로: {data_path}")

# 데이터 구조 확인
print(f"\n--- 데이터 예시 ---")
sample = tool_data[0]
for msg in sample["messages"]:
    role = msg["role"]
    content = msg.get("content", "(null)")
    tool_calls = msg.get("tool_calls", None)
    
    if tool_calls:
        for tc in tool_calls:
            func = tc.get("function", {})
            print(f"  [{role}] → {func.get('name')}({func.get('arguments', '')[:50]})")
    else:
        print(f"  [{role}] {str(content)[:80]}")

In [ ]:
# Tool Calling 메시지를 학습 텍스트로 변환
def format_tool_calling_to_text(item):
    """
    Tool Calling 대화를 학습용 텍스트로 변환합니다.
    tool_calls와 tool 결과를 텍스트로 직렬화합니다.
    """
    formatted_messages = []
    
    for msg in item["messages"]:
        role = msg["role"]
        content = msg.get("content", "")
        tool_calls = msg.get("tool_calls", None)
        
        if role == "tool":
            # tool 응답을 assistant의 관점에서 포함
            formatted_messages.append({
                "role": "user",
                "content": f"[도구 실행 결과] {content}"
            })
        elif tool_calls:
            # tool_calls를 텍스트로 변환
            tc_texts = []
            for tc in tool_calls:
                func = tc.get("function", {})
                func_name = func.get("name", "")
                func_args = func.get("arguments", "{}")
                tc_texts.append(f"[도구 호출] {func_name}({func_args})")
            
            formatted_messages.append({
                "role": "assistant",
                "content": "\n".join(tc_texts)
            })
        else:
            formatted_messages.append({
                "role": role,
                "content": content if content else ""
            })
    
    # Chat Template 적용
    try:
        text = tokenizer.apply_chat_template(
            formatted_messages,
            tokenize=False,
            add_generation_prompt=False
        )
        return text
    except Exception as e:
        return None

# 전체 데이터 변환
formatted_texts = []
failed_count = 0

for item in tool_data:
    text = format_tool_calling_to_text(item)
    if text:
        formatted_texts.append(text)
    else:
        failed_count += 1

dataset = Dataset.from_dict({"text": formatted_texts})

print(f"✅ 데이터 포맷팅 완료")
print(f"📌 성공: {len(formatted_texts)}개")
print(f"📌 실패: {failed_count}개")
print(f"\n--- 포맷팅된 텍스트 예시 ---")
print(dataset[0]["text"][:500])

## 3️⃣ LoRA 설정 및 적용

In [ ]:
# LoRA 설정
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# LoRA 적용
print("⏳ LoRA 어댑터 적용 중...")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print_gpu_memory("LoRA 적용 후")

## 4️⃣ SFTTrainer로 학습 실행

In [ ]:
# 학습 전 응답 저장 (Before)
print("="*60)
print("📋 학습 전 Tool Calling 테스트 (Before)")
print("="*60)

SYSTEM_PROMPT = """당신은 도구를 활용할 수 있는 AI 어시스턴트입니다.
사용 가능한 도구: get_weather(city: str), search_web(query: str), calculate(expression: str), get_stock_price(ticker: str), send_email(to: str, subject: str, body: str)
도구가 필요하면 [도구 호출] 형식으로 호출하세요. 도구가 불필요하면 직접 응답하세요."""

TEST_PROMPTS = [
    "서울 날씨 알려줘",
    "123 곱하기 456은?",
    "삼성전자 주가 확인해줘",
    "안녕하세요! 반갑습니다.",
    "부산이랑 제주 날씨 비교해줘"
]

model.eval()
before_responses = []

for prompt in TEST_PROMPTS:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=200,
            do_sample=False, temperature=1.0
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    before_responses.append(response)
    print(f"\n🔹 질문: {prompt}")
    print(f"   응답: {response[:200]}")

model.train()
print("\n" + "="*60)

In [ ]:
# SFTTrainer 설정
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,                    # Tool Calling은 더 많은 에포크 필요
    per_device_train_batch_size=1,         # RTX 4060 안전 설정
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
    gradient_checkpointing=True,
)

# SFTTrainer 초기화
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("✅ SFTTrainer 초기화 완료")
print(f"📌 epochs: {sft_config.num_train_epochs}")
print(f"📌 batch_size: {sft_config.per_device_train_batch_size}")
print(f"📌 gradient_accumulation: {sft_config.gradient_accumulation_steps}")
print_gpu_memory("학습 시작 전")

In [ ]:
# 학습 실행
print("🚀 Tool Calling 파인튜닝 시작!")
print("="*60)

import time
start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

print("="*60)
print("✅ 학습 완료!")
print(f"📌 소요 시간: {training_time:.1f}초 ({training_time/60:.1f}분)")
print(f"📌 Final Loss: {train_result.training_loss:.4f}")
print(f"📌 Total Steps: {train_result.global_step}")
print_gpu_memory("학습 완료")

## 5️⃣ 학습 전후 Tool Calling 비교

In [ ]:
# 학습 후 응답 생성 및 비교
print("="*60)
print("📊 학습 전후 Tool Calling 비교 (Before vs After)")
print("="*60)

model.eval()
after_responses = []

for i, prompt in enumerate(TEST_PROMPTS):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=200,
            do_sample=False, temperature=1.0
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    after_responses.append(response)
    
    print(f"\n{'='*60}")
    print(f"🔹 질문: {prompt}")
    print(f"\n  [Before] {before_responses[i][:250]}")
    print(f"\n  [After]  {response[:250]}")
    
    # 도구 호출 감지
    has_tool_before = "[도구 호출]" in before_responses[i]
    has_tool_after = "[도구 호출]" in response
    print(f"\n  도구 호출 감지: Before={'✅' if has_tool_before else '❌'} | After={'✅' if has_tool_after else '❌'}")

print(f"\n{'='*60}")

## 6️⃣ 다양한 시나리오 테스트

In [ ]:
# 다양한 Tool Calling 시나리오 테스트
print("="*60)
print("📊 다양한 시나리오 테스트")
print("="*60)

advanced_prompts = [
    # 단일 도구 호출
    ("날씨 조회", "대전 날씨 어때?"),
    ("계산", "원달러 환율 1350원일 때 200달러는 얼마야?"),
    ("검색", "LLM 파인튜닝 최신 기법 검색해줘"),
    ("주가 조회", "NVIDIA 주가 확인해줘"),
    ("이메일", "김팀장에게 프로젝트 완료 보고 이메일 보내줘"),
    
    # 멀티 도구 호출
    ("멀티 도구", "서울이랑 제주도 날씨 비교해줘"),
    ("멀티 도구", "삼성전자랑 SK하이닉스 주가 비교해줘"),
    
    # 도구 불필요
    ("일반 대화", "파이썬이란 무엇인가요?"),
    ("일반 대화", "고마워! 많이 도움이 됐어."),
]

for category, prompt in advanced_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=200,
            do_sample=False, temperature=1.0
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    has_tool = "[도구 호출]" in response
    tool_emoji = "🔧" if has_tool else "💬"
    
    print(f"\n{tool_emoji} [{category}] {prompt}")
    print(f"   → {response[:200]}")

In [ ]:
# 성능 평가 요약
print("\n📊 Tool Calling 성능 평가")
print("="*60)

# 분석
tool_needed = ["날씨 조회", "계산", "검색", "주가 조회", "이메일", "멀티 도구", "멀티 도구"]
tool_not_needed = ["일반 대화", "일반 대화"]

print("""
📌 평가 기준:
  1. 도구가 필요한 질문에 도구를 호출했는가?
  2. 올바른 도구를 선택했는가?
  3. 올바른 인자를 전달했는가?
  4. 도구가 불필요한 질문에 직접 응답했는가?
  5. 멀티 도구 호출이 필요할 때 여러 도구를 호출했는가?

📌 개선 방향:
  - 더 많은 학습 데이터 (500~1000개)
  - 더 다양한 시나리오 포함
  - 에지 케이스 (모호한 질문, 오타 등)
  - 학습 에포크 증가
  - 더 큰 모델 사용 (7B)
""")

## 7️⃣ 모델 저장

In [ ]:
# LoRA 어댑터 저장
save_path = os.path.join(OUTPUT_DIR, "tool_calling_lora")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ 모델 저장 완료: {save_path}")

# 저장 크기 확인
total_size = sum(
    os.path.getsize(os.path.join(root, f))
    for root, dirs, files in os.walk(save_path)
    for f in files
)

print(f"\n--- 저장된 파일 ---")
for root, dirs, files in os.walk(save_path):
    for f in files:
        fp = os.path.join(root, f)
        size = os.path.getsize(fp)
        print(f"  📄 {f}: {size/1024/1024:.2f} MB")

print(f"\n📌 전체 어댑터 크기: {total_size/1024/1024:.1f} MB")
print(f"📌 이 어댑터를 base 모델에 로드하면 Tool Calling 가능!")

In [ ]:
# 어댑터 로드 방법 안내
print("\n📋 저장된 어댑터 사용 방법")
print("="*60)

print(f"""
# 어댑터 로드 코드
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Base 모델 로드
base_model = AutoModelForCausalLM.from_pretrained(
    "{MODEL_NAME}",
    quantization_config=bnb_config,
    device_map="auto"
)

# LoRA 어댑터 로드
model = PeftModel.from_pretrained(
    base_model,
    "{save_path}"
)

# 추론
tokenizer = AutoTokenizer.from_pretrained("{save_path}")
""")

In [ ]:
# GPU 메모리 정리
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print_gpu_memory("메모리 정리 후")
print("✅ GPU 메모리 정리 완료")

## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | 내용 |
|------|------|
| Tool Calling 데이터 포맷 | 멀티턴 대화를 Chat Template으로 변환 |
| QLoRA 파인튜닝 | 3B 모델 4bit + LoRA로 RTX 4060에서 학습 |
| 학습 전후 비교 | 도구 호출 능력의 변화 확인 |
| 다양한 테스트 | 단일/멀티/미호출 시나리오 모두 테스트 |

### Tool Calling 파인튜닝 전체 파이프라인 요약

```
1. 도구 정의 (JSON Schema)
2. 학습 데이터 생성 (GPT-4o-mini 합성)
3. 데이터 검증 및 필터링
4. Chat Template 포맷팅
5. QLoRA 파인튜닝
6. Before/After 비교 테스트
7. 어댑터 저장 및 배포
```

### 핵심 포인트

- 🎯 **Tool Calling은 데이터 품질이 핵심** - 다양하고 정확한 데이터 필요
- 🎯 **도구 호출/미호출 균형** - 둘 다 잘 구분해야 합니다
- 🎯 **더 많은 에포크** - Tool Calling은 패턴 학습에 더 많은 반복 필요
- 🎯 **실무에서는 500+ 데이터** 필요, 지속적 개선 중요
- 🎯 **RTX 4060에서 3B QLoRA**로 충분히 학습 가능

### Part 4 전체 정리

| 세션 | 내용 |
|------|------|
| 17. Continuous Pretraining | 도메인 텍스트로 지식 주입 |
| 18. Instruction Tuning | 지시-응답 학습으로 지시 수행 능력 향상 |
| 19. LoRA vs FFT 이론 | 파라미터, 메모리, 속도 비교 |
| 20. LoRA vs FFT 실전 | 동일 데이터로 성능 비교 |
| 21. Unsloth | 2배 빠른 학습, GGUF 변환 |
| 22. Tool Calling 개념 | OpenAI API 기반 도구 호출 |
| 23. Tool Calling 데이터 | GPT-4o-mini로 합성 데이터 생성 |
| 24. Tool Calling 파인튜닝 | sLLM에 도구 호출 능력 부여 |

### 다음 파트

- ➡️ **Part 5**: 강화학습 (PPO, DPO, GRPO)
- ➡️ 파인튜닝 후 강화학습으로 모델 품질을 더욱 향상시킵니다!